In [ ]:
# def Training(training_data, training_labels, test_data, test_labels, epoch = 100, random_seed = 42, nn_hidden_layers = 7, learning_rate = 0.001):
#     np.random.seed(random_seed)
#     training_data = training_data.to_numpy()
#     encoder = OneHotEncoder(sparse_output=False)
#     Y = encoder.fit_transform(training_labels.to_numpy().reshape(-1, 1))
#     Biases = np.zeros(nn_hidden_layers)
#     output_bias = np.ones(Y.shape[1])
#     Hidden_layer = np.random.randn(nn_hidden_layers,training_data.shape[1]) * np.sqrt(2/training_data.shape[1])
#     output_layer = np.random.randn(Y.shape[1],nn_hidden_layers) * np.sqrt(2/nn_hidden_layers)
#     accuracies = []
#     losses = []
#     X_test = test_data.to_numpy()
#     y_test = test_labels.to_numpy().ravel().astype(int)
#     test_accuracies = []
#     test_losses = []
#     for e in range(epoch):
#         counter = 0
#         total_loss = 0
#         correct = 0
#
#         for row, target in zip(training_data, Y):
#             counter += 1
#             into_hidden = (Hidden_layer @ row) + Biases
#             activated_hidden = ReLU(into_hidden)
#             into_output = (output_layer @ activated_hidden) + output_bias
#             probs = Softmax(into_output)
#             total_loss += -log(probs @ target)
#             #total_loss += np.sum((probs - target) ** 2)
#             if np.argmax(probs) == np.argmax(target):
#                 correct += 1
#             error_out = probs - target
#             #d = 2 * (probs - target)
#             #error_out = probs * (d - np.sum(d * probs))
#
#             error_hidden = output_layer.T @ error_out
#             error_hidden = error_hidden * (into_hidden > 0)
#
#             grad_output_bias  = error_out
#             grad_output_layer = np.outer(error_out, activated_hidden) + output_layer * 0.008
#             grad_hidden_bias  = error_hidden
#             grad_hidden_layer = np.outer(error_hidden, row) + Hidden_layer * 0.028
#
#             output_bias  -= learning_rate * grad_output_bias
#             output_layer -= learning_rate * grad_output_layer
#             Biases       -= learning_rate * grad_hidden_bias
#             Hidden_layer -= learning_rate * grad_hidden_layer
#
#         losses.append(total_loss/len(training_data))
#         accuracies.append(correct /len(training_data))
#         print(f"epoch {e+1:3d}/{epoch}  row {counter:5d}  "
#                   f"loss {total_loss/counter:.4f}  acc {correct/counter:.4f}",
#                   end="\r", flush=True)
#
#
#
#     X_test = test_data.to_numpy()
#     y_test = test_labels.to_numpy().ravel().astype(int)
#
#     test_loss = 0
#     test_correct = 0
#     for row, true_digit in zip(X_test, y_test):
#         into_hidden = (Hidden_layer @ row) + Biases
#         activated_hidden = ReLU(into_hidden)
#         into_output = (output_layer @ activated_hidden) + output_bias
#         probs = Softmax(into_output)
#
#         test_loss += -np.log(probs[true_digit])
#         if np.argmax(probs) == true_digit:
#             test_correct += 1
#
#     test_loss /= len(X_test)
#     test_accuracy = test_correct / len(X_test)
#     print(f"\ntest   loss {test_loss:.4f}  acc {test_accuracy:.4f}")
#
"""
Implemented a class from the old training loop also added more error handling for future use. Also updated the naming conventions to make more sense.
"""

# class MLP:
#     """A two-layer multi-layer perceptron trained with stochastic gradient descent."""
#
#     def __init__(self, n_hidden=7, learning_rate=0.001, lam_hidden=0.028,
#                  lam_output=0.008, loss="cross_entropy", random_seed=42):
#         self.set_params(
#             n_hidden=n_hidden,
#             learning_rate=learning_rate,
#             lam_hidden=lam_hidden,
#             lam_output=lam_output,
#             loss=loss,
#             random_seed=random_seed,
#         )
#         self.history = {"train_loss": [], "train_acc": [],
#                         "test_loss": [], "test_acc": []}
#         self.classes_ = None
#         self.Hidden_layer = None
#
#
#     def set_params(self, **kwargs):
#         """Change any parameter key:value style"""
#         allowed = {"n_hidden", "learning_rate", "lam_hidden",
#                    "lam_output", "loss", "random_seed"}
#         for key, value in kwargs.items():
#             if key not in allowed:
#                 raise ValueError(f"Unknown parameter: {key}. Allowed: {sorted(allowed)}")
#             setattr(self, key, value)
#         if self.loss not in ("cross_entropy", "mse"):
#             raise ValueError("loss must be 'cross_entropy' or 'mse'")
#         return self
#
#     def get_params(self):
#         return {key: getattr(self, key) for key in
#                 ("n_hidden", "learning_rate", "lam_hidden",
#                  "lam_output", "loss", "random_seed")}
#
#     def _init_weights(self, n_features, n_classes):
#         np.random.seed(self.random_seed)
#         self.Hidden_layer = np.random.randn(self.n_hidden, n_features) * np.sqrt(2 / n_features)
#         self.Biases = np.zeros(self.n_hidden)
#         self.output_layer = np.random.randn(n_classes, self.n_hidden) * np.sqrt(2 / self.n_hidden)
#         self.output_bias = np.zeros(n_classes)
#
#     @staticmethod
#     def _as_array(data):
#         return data.to_numpy() if hasattr(data, "to_numpy") else np.asarray(data)
#
#
#     def _forward(self, row):
#         into_hidden = self.Hidden_layer @ row + self.Biases
#         activated_hidden = ReLU(into_hidden)
#         into_output = self.output_layer @ activated_hidden + self.output_bias
#         probs = Softmax(into_output)
#         return into_hidden, activated_hidden, probs
#
#     def _loss(self, probs, target):
#         if self.loss == "cross_entropy":
#             return -np.log(probs @ target)
#         return np.sum((probs - target) ** 2)
#
#     def _output_error(self, probs, target):
#         if self.loss == "cross_entropy":
#             return probs - target
#         d = 2 * (probs - target)
#         return probs * (d - np.sum(d * probs))
#
#
#     def fit(self, X, y, epochs=100, test_data=None, test_labels=None, verbose=True):
#         X = self._as_array(X)
#         encoder = OneHotEncoder(sparse_output=False)
#         Y = encoder.fit_transform(self._as_array(y).reshape(-1, 1))
#         self.classes_ = encoder.categories_[0]
#
#         self._init_weights(X.shape[1], Y.shape[1])
#         self.history = {"train_loss": [], "train_acc": [],
#                         "test_loss": [], "test_acc": []}
#
        # for e in range(epochs):
        #     total_loss = 0.0
        #     correct = 0
        #
        #     for row, target in zip(X, Y):
        #         into_hidden, activated_hidden, probs = self._forward(row)
        #
        #         total_loss += self._loss(probs, target)
        #         if np.argmax(probs) == np.argmax(target):
        #             correct += 1
        #
        #         error_out = self._output_error(probs, target)
        #         error_hidden = (self.output_layer.T @ error_out) * (into_hidden > 0)
        #
        #         grad_output_layer = np.outer(error_out, activated_hidden) + self.lam_output * self.output_layer
        #         grad_hidden_layer = np.outer(error_hidden, row) + self.lam_hidden * self.Hidden_layer
        #
        #         self.output_bias -= self.learning_rate * error_out
        #         self.output_layer -= self.learning_rate * grad_output_layer
        #         self.Biases -= self.learning_rate * error_hidden
        #         self.Hidden_layer -= self.learning_rate * grad_hidden_layer
        #
        #     self.history["train_loss"].append(total_loss / len(X))
        #     self.history["train_acc"].append(correct / len(X))
        #
        #     line = (f"epoch {e + 1:3d}/{epochs}  "
        #             f"loss {self.history['train_loss'][-1]:.4f}  "
        #             f"acc {self.history['train_acc'][-1]:.4f}")
        #
        #     if test_data is not None and test_labels is not None:
        #         t_loss, t_acc, _ = self.evaluate(test_data, test_labels)
        #         self.history["test_loss"].append(t_loss)
        #         self.history["test_acc"].append(t_acc)
        #         line += f"  |  test loss {t_loss:.4f}  test acc {t_acc:.4f}"
        #
        #     if verbose:
        #         print(line + " " * 8, end="\r", flush=True)
        #
        # if verbose:
        #     print()
        # return self
#
#
#     def predict(self, X):
#         if self.Hidden_layer is None:
#             raise RuntimeError("Call fit() before predict().")
#         X = self._as_array(X)
#         return np.array([self.classes_[np.argmax(self._forward(row)[2])] for row in X])
#
#     def evaluate(self, X, y):
#         """Evaluate the training weights"""
#         X = self._as_array(X)
#         y = self._as_array(y).ravel()
#
#         label_to_slot = {label: i for i, label in enumerate(self.classes_)}
#         total_loss = 0.0
#         correct = 0
#         predictions = []
#
#         for row, true_label in zip(X, y):
#             _, _, probs = self._forward(row)
#             slot = label_to_slot[true_label]
#
#             target = np.zeros(len(self.classes_))
#             target[slot] = 1.0
#             total_loss += self._loss(probs, target)
#
#             predicted = np.argmax(probs)
#             predictions.append(self.classes_[predicted])
#             if predicted == slot:
#                 correct += 1
#
#         return total_loss / len(X), correct / len(X), np.array(predictions)
#
#
#
#
# mlp = MLP(n_hidden=2, learning_rate=0.001, lam_hidden=0.028, lam_output=0.008)
# mlp.fit(train_in, train_out, epochs=90, test_data=test_in, test_labels=test_out)
#
# loss, acc, predictions = mlp.evaluate(test_in, test_out)
# print(f"final test  loss {loss:.4f}  acc {acc:.4f}")